# Cross-age Generalization in Native Language Identification

This notebook analyzes how well NLI models generalize across age groups (training on adults, testing on children).

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score

# Assuming our project modules are available
import sys
sys.path.append('..')

from config import LANGUAGES
from models.classifiers import HubertClassifier
from models.trainer import ModelTrainer

In [ ]:
def evaluate_cross_age_generalization(model, adult_test_loader, child_test_loader, device='cpu'):
    """
    Evaluate model generalization from adults to children
    """
    model.eval()
    
    def evaluate_loader(loader, group_name):
        all_targets = []
        all_predictions = []
        
        with torch.no_grad():
            for data, targets in loader:
                data, targets = data.to(device), targets.to(device)
                outputs = model(data)
                _, predicted = torch.max(outputs, 1)
                
                all_targets.extend(targets.cpu().numpy())
                all_predictions.extend(predicted.cpu().numpy())
        
        accuracy = accuracy_score(all_targets, all_predictions)
        print(f"{group_name} Accuracy: {accuracy:.4f}")
        return accuracy, all_targets, all_predictions
    
    # Evaluate on adults (in-domain)
    adult_accuracy, adult_targets, adult_predictions = evaluate_loader(adult_test_loader, "Adult")
    
    # Evaluate on children (out-of-domain)
    child_accuracy, child_targets, child_predictions = evaluate_loader(child_test_loader, "Child")
    
    return {
        'adult_accuracy': adult_accuracy,
        'child_accuracy': child_accuracy,
        'adult_targets': adult_targets,
        'adult_predictions': adult_predictions,
        'child_targets': child_targets,
        'child_predictions': child_predictions
    }

In [ ]:
def compare_feature_robustness(mfcc_results, hubert_results):
    """
    Compare robustness of MFCC vs HuBERT features for cross-age generalization
    """
    # Extract accuracies
    mfcc_adult_acc = mfcc_results['adult_accuracy']
    mfcc_child_acc = mfcc_results['child_accuracy']
    hubert_adult_acc = hubert_results['adult_accuracy']
    hubert_child_acc = hubert_results['child_accuracy']
    
    # Calculate drops in performance
    mfcc_drop = mfcc_adult_acc - mfcc_child_acc
    hubert_drop = hubert_adult_acc - hubert_child_acc
    
    # Visualization
    features = ['MFCC', 'HuBERT']
    adult_accs = [mfcc_adult_acc, hubert_adult_acc]
    child_accs = [mfcc_child_acc, hubert_child_acc]
    
    x = np.arange(len(features))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    rects1 = ax.bar(x - width/2, adult_accs, width, label='Adults', color='skyblue')
    rects2 = ax.bar(x + width/2, child_accs, width, label='Children', color='lightcoral')
    
    ax.set_ylabel('Accuracy')
    ax.set_title('Cross-age Generalization Performance')
    ax.set_xticks(x)
    ax.set_xticklabels(features)
    ax.legend()
    ax.set_ylim(0, 1)
    
    # Add value labels
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.3f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha='center', va='bottom')
    
    autolabel(rects1)
    autolabel(rects2)
    
    plt.tight_layout()
    plt.show()
    
    # Print analysis
    print("\nCross-age Generalization Analysis:")
    print(f"MFCC - Adult accuracy: {mfcc_adult_acc:.4f}, Child accuracy: {mfcc_child_acc:.4f}, Drop: {mfcc_drop:.4f}")
    print(f"HuBERT - Adult accuracy: {hubert_adult_acc:.4f}, Child accuracy: {hubert_child_acc:.4f}, Drop: {hubert_drop:.4f}")
    
    if mfcc_drop < hubert_drop:
        print("MFCC features show better cross-age generalization.")
    else:
        print("HuBERT features show better cross-age generalization.")

In [ ]:
# Placeholder for actual experiment
# In practice, you would:

"""
# 1. Load adult and child datasets
# adult_train_loader, adult_val_loader, adult_test_loader = load_adult_data()
# child_test_loader = load_child_data()

# 2. Train models on adult data
# mfcc_model = MFCCClassifier(num_classes=len(LANGUAGES))
# hubert_model = HubertClassifier(num_classes=len(LANGUAGES))

# mfcc_trainer = ModelTrainer(mfcc_model)
# mfcc_trainer.train(adult_train_loader, adult_val_loader)

# hubert_trainer = ModelTrainer(hubert_model)
# hubert_trainer.train(adult_train_loader, adult_val_loader)

# 3. Evaluate cross-age generalization
# mfcc_results = evaluate_cross_age_generalization(mfcc_model, adult_test_loader, child_test_loader)
# hubert_results = evaluate_cross_age_generalization(hubert_model, adult_test_loader, child_test_loader)

# 4. Compare feature robustness
# compare_feature_robustness(mfcc_results, hubert_results)
"""

## Analysis of Cross-age Generalization

When models trained on adult speech are tested on children's speech, we typically observe a drop in performance. This is due to several factors:

1. **Vocal Tract Differences**: Children have shorter vocal tracts, affecting formant frequencies
2. **Articulation Patterns**: Children may not have fully developed articulation skills
3. **Phonological Development**: Children's phonological systems are still developing
4. **Speech Rate**: Children often speak at different rates than adults

The comparison between MFCC and HuBERT features can reveal which representation is more robust to these variations. Self-supervised features like HuBERT might capture more invariant representations that generalize better across age groups.